## In this notebook we will
1. Perform downconversion and downsampling of a long, real signal using FIR filtering.
2. Look at the overlap-save technique will be demonstrated to filter several small chunks of data in a continuous fashion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import firwin, lfilter, lfilter_zi

In [ ]:
plt.rcParams.update({
    # Figure & Resolution
    'figure.figsize': (3, 1.5),      # Standard rectangular size
    'figure.dpi': 100,             # High resolution for saving
    'savefig.dpi': 300,            # High resolution for exported images
    'savefig.bbox': 'tight',       # Removes unnecessary white space around the plot
    
    # Fonts & Text
    'font.family': 'sans-serif',   # Use serif for traditional journals, sans-serif for modern
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,               # Base font size
    'axes.titlesize': 10,          # Title size
    'axes.labelsize': 10,          # X and Y label size
    'xtick.labelsize': 10,         # Tick label size
    'ytick.labelsize': 10,         # Tick label size
    'legend.fontsize': 10,         # Legend text size
    
    # Axes & Spines (The "Clean" Look)
    'axes.spines.top': False,      # Remove top bounding box line
    'axes.spines.right': False,    # Remove right bounding box line
    'axes.linewidth': 1.1,         # Slightly thicker axes lines
    'axes.grid': False,            # Default to no grid (turn on manually if needed)
    
    # Ticks
    'xtick.direction': 'in',       # Ticks point inward
    'ytick.direction': 'in',       # Ticks point inward
    'xtick.major.size': 6,         # Major tick length
    'xtick.major.width': 1.2,      # Major tick thickness
    'ytick.major.size': 6,         # Major tick length
    'ytick.major.width': 1.2,      # Major tick thickness
    
    # Lines & Markers
    'lines.linewidth': 1.5,        # Thicker lines for visibility
    'lines.markersize': 4,         # Standard marker size

    # Legend
    'legend.frameon': False,       # Remove the box around the legend
    'legend.loc': 'best'           # Automatically place legend out of the way
})

### We once again start from a long, timestream that we assume we dont have access to all at once.

In [ ]:
#generate some spectra

Norig = 256
extend = 16
N = Norig * extend
X = np.zeros(N//2+1,dtype='complex128')
Xc = np.zeros(N,dtype='complex128') #analytic
k0 = 32 * extend + 5 #Will not be at the center of any channel of smaller FFT sizes. 
dk = 8 * extend
triangle = np.arange(0,2*dk)/N + 1j*np.arange(0,2*dk)/N #triangular real and imag parts (not-symmetric)

X[k0-dk:k0+dk]=triangle 
x=np.fft.irfft(X) #note irfft. -ve freqs will be conjugate of +ve freqs.  generates a real signal


Xc[k0-dk:k0+dk]=2*triangle # ignore -ve freqs and double +ve freqs
x2=np.fft.ifft(Xc) #note ifft.  generates a complex signal

freqs = np.fft.fftshift(np.fft.fftfreq(N))*N
# print(freqs)
fig=plt.gcf()
fig.set_size_inches(12,2)
plt.plot(freqs,np.fft.fftshift(np.fft.fft(x2).real),label='analytic re' )
plt.plot(freqs,np.fft.fftshift(np.fft.fft(x2).imag),label='analytic im' ,ls='dashed')
plt.xlabel("freqs")
plt.title("analytic spectra re and im parts")
plt.legend()


print("x type:", x.dtype, "x2 type:", x2.dtype)

In [ ]:
x_rotated = x*np.exp(-2j*np.pi*np.arange(N)*k0/N)
plt.title("spectra of real signal d/c")
fig=plt.gcf()
fig.set_size_inches(12,2)
plt.plot(freqs,np.fft.fftshift(np.fft.fft(x_rotated).real))
plt.plot(freqs,np.fft.fftshift(np.fft.fft(x_rotated).imag),ls='dotted')
plt.axvspan(-dk,dk,color='red',alpha=0.1,label='Ideal filter')
plt.legend()

We want to snip the highlighted part of the spectrum. In this simulation, we have access to the entire big block of data. If we wanted, we can just extract the part we want from the array. 

In real-life, that's almost never the case. Imagine filtering data continuously on a car FM radio. We're not going to take one gigantic 10-hour long FFT. Instead, we must think about how to filter continuously in time domain.

### We can use scipy to help us design a filter of choice

In [ ]:
#first lets design a filter
cutoff = 8/256 # 8 was the one-sided bandwidth in units of original FFT length
print("filter +ve freq cutoff in units of sampling freq", cutoff)
filter_size = 128
h = firwin(filter_size, cutoff=cutoff, window='hamming',fs=1)
fig=plt.gcf()
fig.set_size_inches(12,2)
plt.subplot(121)
plt.title("Filter coefficients")
plt.plot(h)

plt.subplot(122)
plt.title("Filter response power")
filter_response = np.fft.fft( np.hstack([ h, np.zeros( 10*len(h) ) ]) ) 
#stack some zeros to increase freq resolution
freqs = np.fft.fftfreq(len(filter_response))
plt.semilogy( np.fft.fftshift(freqs),  np.fft.fftshift(np.abs(filter_response)**2))
plt.xlim(-cutoff-0.02, cutoff+0.02)
plt.axvline(cutoff, ls='dashed', c='black')
plt.axvline(-cutoff, ls='dashed', c='black',label='cutoff')
plt.axhline(0.25,ls='dotted', c='red',label='-6 dB point')
plt.xlabel("Freqs (fraction of 2*Nyquist)")
plt.ylim(1e-7,1)
plt.legend()


Scipy `firwin` returns causal filters. This is what we use in real-life because a filter never has access to future data.

Usual filtering equation goes

$y[n] = \sum_m x[n - m] h[m]$

Our sim data was generated using an FFT and is therefore sitting in a circular buffer. Let us roll it so that left-to-right is a linear arrow in time. Then we'll filter it in chunks of 256.

In [ ]:
x_rotated_linear = np.fft.fftshift(x_rotated)
plt.plot(x_rotated_linear.real)
plt.title("left->right is forward in time")

We will now filter our data in blocks of size 256 instead of doing all at once using a technique called Overlap-save.

The idea is simple. Say we want to filter a data in blocks of size 9N with a filter of size N. To do so we make an input buffer of size 10N and a filter buffer of size 10N. Then we fill the first N samples in input buffer with zeros and next 9N with the first block of data, and we fill first N samples in the filter buffer with the actual filter coefficients and the rest with zeros. Then we convolve the two with FFTs but remeber that convolution with FFTs is circular. 

So, first N samples in the OUTPUT buffer come from N samples at the END of the INPUT buffer - i.e. the future. This is not causal. So we discard those samples. Valid output starts from sample N. So we get 9N valid output samples, 0 to 9N-1. Then we move on to next input block of 9N samples.

The output at time 9N depends on samples 9N-1, 9N-2,...8N, in the PREVIOUS input block. So in the input buffer, we fill the first N numbers with last N samples of the previous input block to stitch them together and make the filtering continuous in time. And then we fill the remaining 9N numbers in the input buffer with the second block of data. 

Once again upon filtering, we discard the first N samples. Sample number N in SECOND output block is now the output sample 9N, and we get another 9N valid output samples from 9N to 18N-1. And we keep repeating this.

![Alternative text](../images/overlap_save.jpg)

In [ ]:
blocksize = 512 #say data comes in blocks of size 512
nblocks = x_rotated_linear.shape[0]//blocksize
L = filter_size
x = x_rotated_linear[:nblocks*blocksize].reshape(nblocks, blocksize) #snip the last remainder (none in our case)

#create an input buffer with extra padding to put past data
inp = np.zeros((nblocks, blocksize + L), dtype='complex128')
inp[:, L:] = x[:, :] #fill current
inp[1:, :L] = x[:-1, -L:] #fill first padded chunk with past data data from filter-length-long bit of previous block

hf = np.fft.fft(np.hstack([h, np.zeros(blocksize)])) #create filter spectrum
filt_inp = np.fft.ifft( np.fft.fft(inp,axis=1) * hf[None,:], axis=1) #filter the input

#extract output by discarding output corresponding to padded blocks
out = filt_inp[:, L:]
out = np.ravel(out)

If things have gone right, we've correctly filtered out signal and removed those pesky negative frequency images. We should be safe to downsample now by a factor of $f_s/BW$.

In [ ]:
dsamp = Norig//(2*8)
print("downsampling by a factor of", dsamp)
out2 = out[::dsamp]


### Is the spectrum going to line up with a triangle that's nice and uncorrupted?

drumroll...

In [ ]:
# plt.plot(np.fft.fftshift(np.abs(np.fft.fft(out2))))
plt.plot(np.fft.fftshift(np.abs(np.fft.fft(out2))))

### Whoops. why has it started to roll-off?

Our signal is a straight line. Refresh your memory by looking at past notebooks. The roll-off is because we put our filter -3 dB cutoff right at the edge of our signal. Ideally, we'd like the passband to be flat throughout the signal of interest.

### EXERCISE
Re-run the filtering with twice the original cutoff and see what happens

Congratulations! You've successfully implemented a general purpose overlap-save FIR filter.